# MNIST LoRA Ablation

Compares the paper's per-requirement MNIST LoRAs with the shared `Allreq` and `Alldata` MNIST ablations using validity-aware failure efficiency.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
COLAB_ROOT = Path("/content/rbt4dnn-seminar")
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "data/requirements.csv").exists()), None)
if ROOT is None:
    ROOT = COLAB_ROOT
    if (ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=True)
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

for module in [
    "shared",
    "mnist",
    "precondition_validity",
    "cost_analysis",
    "mnist_lora_ablation",
    "mnist_shared_generator",
]:
    sys.modules.pop(module, None)

EXPERIMENT = ROOT / "experiments" / "mnist-lora-ablation"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT))
print(ROOT)
commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"], text=True
).strip()
print(commit)

In [ ]:
from mnist_lora_ablation import build_rows, write_results
from shared import requirement_rows

results_path, summary_path = write_results(ROOT)
rows = build_rows(requirement_rows(ROOT))

print("requirement method | estimated valid failures per 1000 | relative to per-requirement LoRA")
for row in rows:
    print(
        f"{row['requirement']} {row['method']} | "
        f"{row['estimated_valid_failures_per_1000']} | {row['relative_to_per_requirement_lora']}"
    )
print(results_path)
print(summary_path)

This is the strongest no-retraining extension because it directly tests whether one shared MNIST LoRA strategy is competitive with one LoRA per requirement.